[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesamKhallaf73/pytorch-ground-up-notebooks/blob/main/notebooks/L14.ipynb)

# Lecture 14 — DataLoader, Transforms & Augmentation

*The machine that feeds batches to your training loop — and the pipeline that makes every sample model-ready, and a little different each epoch.*

**PyTorch from Ground Up · Volume I · Chapters 24–25** — companion notebook.

**How to use it:** run the cells from top to bottom. Before each code cell, *predict* what it will print — then run it and compare with the expected output (click ▸ to open it). Every exercise from the lecture is at the end.

By the end you will be able to:
- Wrap a Dataset in a DataLoader and iterate an epoch of batches.
- Predict the number of batches and the size of the last one.
- Use `drop_last`, `num_workers` and `collate_fn` appropriately.
- Write the complete two-loop training structure.
- Chain transforms with `Compose` in the right order.
- Separate training augmentation from deterministic evaluation transforms, and write a custom transform.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print('PyTorch', torch.__version__)

## 01 · From one sample to a batch

A Dataset gives **one sample**; a model wants a **batch**. The DataLoader bridges the gap. It draws samples — shuffled if you like — stacks them into batches and hands them to the loop one at a time.

<sub>Source: lecture slides 3–11</sub>

**`loader_basics.py`** — predict the output, then run.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
x = torch.randn(100, 5)
y = torch.randint(0, 3, (100,))
ds = TensorDataset(x, y)
loader = DataLoader(ds, batch_size=16, shuffle=True)

for batch_x, batch_y in loader:
    print(batch_x.shape, batch_y.shape)
    break      # just the first batch

<details><summary>▸ Expected output</summary>

```
torch.Size([16, 5]) torch.Size([16])
```
</details>

🧪 *Interactive lab in the app:* **Deal out an epoch** — try it there to see this idea move.

**`epochs.py`** — predict the output, then run.

In [ ]:
for epoch in range(3):
    for batch_idx, (bx, by) in enumerate(loader):
        pass     # training step here
    print(f"Epoch {epoch}: {batch_idx+1} batches")

<details><summary>▸ Expected output</summary>

```
Epoch 0: 7 batches
Epoch 1: 7 batches
Epoch 2: 7 batches
```
</details>

## 02 · The settings that matter

`drop_last=True` throws away the short final batch — **fine for training, wrong for evaluation**. Tiny batches upset BatchNorm. In training the dropped samples return next epoch; in evaluation you want every sample scored.

<sub>Source: lecture slides 12–21</sub>

> ⚠️ **Watch out — Workers need a guard**  
> On Google Colab, `num_workers=2` is usually safe; higher values can cause memory issues or crashes on free-tier machines.
>
> On Windows and macOS, always guard your training script with `if __name__ == "__main__":` when using workers, or you will get a multiprocessing error.

**`pad_collate.py`** — predict the output, then run.

In [ ]:
def pad_collate(batch):
    # batch is a list of (feature, label) tuples
    features, labels = zip(*batch)
    # Pad features to the longest in this batch
    padded = torch.nn.utils.rnn.pad_sequence(
        features, batch_first=True)
    return padded, torch.tensor(labels)

loader = DataLoader(ds, batch_size=4,
                    collate_fn=pad_collate)

## 03 · The complete training loop

The only new line since Lecture 03 is **the inner `for` loop**. Model, loss and optimizer plug in exactly as before. This is the structure you’ll see in every PyTorch project.

<sub>Source: lecture slides 22–24</sub>

**`train_loop.py`** — predict the output, then run.

In [ ]:
ds = TensorDataset(x, y)
loader = DataLoader(ds, batch_size=16, shuffle=True)
model = nn.Linear(5, 3)
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(10):
    for bx, by in loader:
        loss = loss_fn(model(bx), by)
        opt.zero_grad()
        loss.backward()
        opt.step()
    print(f"Epoch {epoch}  Loss: {loss.item():.4f}")

## 04 · Why transforms: the pipeline and Compose

Raw data is **never model-ready**. Transforms close the gap — and can invent new training data. Sizes differ, pixels run 0–255, channels may be in the wrong order. Random flips, crops and colour shifts multiply your dataset without collecting a single new sample.

<sub>Source: lecture slides 26–33</sub>

**`compose.py`** — predict the output, then run.

In [ ]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((32, 32)),            # fixed
    transforms.RandomHorizontalFlip(),      # random
    transforms.ToTensor(),                  # fixed
    transforms.Normalize(                   # fixed
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]),
])

> ⚠️ **Watch out — ToTensor before Normalize**  
> `ToTensor` must come before `Normalize`.
>
> `Normalize` works on tensors, not PIL images — and its mean and std assume pixels already scaled to [0, 1], which is `ToTensor`’s job.

## 05 · Augmentation: seeing more than you have

Augmentation: the model **never sees the exact same image twice**. One of the simplest, most effective ways to reduce overfitting.

<sub>Source: lecture slides 34–38</sub>

🧪 *Interactive lab in the app:* **Augmentation playground** — try it there to see this idea move.

## 06 · Train vs eval transforms

The critical rule: **augmentation is for training only**. Evaluation must be deterministic.

<sub>Source: lecture slides 39–44</sub>

_Stand-in: the next listing uses names from your own project (a model, a data loader…). This cell defines small ones so it runs here._

In [ ]:
from torchvision import transforms
from torch.utils.data import Dataset
class MyDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
    def __len__(self):
        return 8
    def __getitem__(self, i):
        from PIL import Image
        img = Image.new('RGB', (300, 260), (i * 30, 90, 160))
        return (self.transform(img) if self.transform else img), i % 2
root = 'unused'


**`eval_transform.py`** — predict the output, then run.

In [ ]:
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([.485,.456,.406],
                         [.229,.224,.225]),
])

train_ds = MyDataset(root, transform=train_transform)
eval_ds  = MyDataset(root, transform=eval_transform)

> ⚠️ **Watch out — Augmentation at eval time**  
> Using augmentation during evaluation is a subtle but serious bug. The random transforms make predictions non-deterministic and inflate variance in your metrics.
>
> Always define two separate transforms and use the right one for each split.

## 07 · Beyond images, and your own transforms

Not just images: **a transform is any callable** from raw data to a model-ready tensor. Tabular: normalise columns. Text: tokenise and pad. Audio: compute a spectrogram.

<sub>Source: lecture slides 45–50</sub>

**`gaussian_noise.py`** — predict the output, then run.

In [ ]:
class AddGaussianNoise:
    def __init__(self, mean=0.0, std=0.1):
        self.mean = mean
        self.std  = std
    def __call__(self, tensor):
        noise = torch.randn_like(tensor) * self.std
        return tensor + noise + self.mean

train_transform = transforms.Compose([
    transforms.ToTensor(),
    AddGaussianNoise(std=0.05),
    transforms.Normalize([.5], [.5]),
])

## The rule — worth remembering

**DataLoader — six rules**

1. `DataLoader(dataset, batch_size, shuffle)` turns a Dataset into an iterable of batches.
2. Each iteration yields a tuple of tensors — one per element the Dataset returns.
3. `shuffle=True` for training, `shuffle=False` for evaluation.
4. `drop_last=True` discards the final small batch — useful for BatchNorm, avoid during eval.
5. `num_workers > 0` loads data in parallel. Start with 2.
6. Use `collate_fn` when samples have different shapes and need custom merging.

**Transforms — six rules**

1. `Compose` chains transforms into a single callable pipeline.
2. Fixed transforms (`Resize`, `ToTensor`, `Normalize`) run always. Random transforms (Flip, Crop, Jitter) run only during training.
3. Order matters: spatial transforms on PIL images → `ToTensor` → tensor transforms → `Normalize`.
4. Define two pipelines: `train_transform` (with augmentation) and `eval_transform` (without).
5. Augmentation reduces overfitting by showing the model different views of the same data.
6. Any object with `__call__` can be a transform — custom transforms compose like built-in ones.

## Pause and try

Work each exercise in the empty cell below it. Open the worked answer only after you have tried.

### Exercise 01 · Ch 24 · Ex 1

A `TensorDataset` of 200 samples with 8 features and labels from 0 to 4, in a DataLoader with `batch_size=32, shuffle=True`. How many batches per epoch?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

⌈200 / 32⌉ = **7**: six full batches of 32 (192 samples) and one batch of 8.
</details>

### Exercise 02 · Ch 24 · Ex 3

Two DataLoaders from the same Dataset, one with `shuffle=True`, one with `shuffle=False`. Print the first batch’s labels from each. Are they the same?

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

No. The unshuffled loader returns the first samples in dataset order, every time; the shuffled one returns a random selection that changes each epoch.
</details>

### Exercise 03 · Ch 24 · Ex 6

A Dataset whose samples have random lengths between 5 and 15. Iterate with the default DataLoader — what error do you get? Then write a `collate_fn` that pads each batch to its longest sample.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`RuntimeError: stack expects each tensor to be equal size…` — default collation can only stack equal shapes. The fix is the `pad_collate` function above, using `torch.nn.utils.rnn.pad_sequence(features, batch_first=True)`.
</details>

### Exercise 04 · Try it — flip and normalize a fake image

Create a random (3, 64, 64) tensor as a fake image. Apply `RandomHorizontalFlip(p=1.0)` (forced flip) and compare with the original — which axis is reversed? Then apply `Normalize([0.5]*3, [0.5]*3)` and check the value range.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

The **last axis (width)** is reversed, so `original[0, :, 0]` equals `flipped[0, :, -1]`: the first column became the last. `Normalize` with mean 0.5 and std 0.5 maps values from [0, 1] to **[−1, 1]**.
</details>

### Exercise 05 · Ch 25 · Ex 6

What happens if you apply `Normalize` before `ToTensor`? Try it and explain the error.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

It raises a TypeError — `Normalize` received a PIL Image, and it only works on tensors. `ToTensor` must come first, both to convert the type and to scale the pixels to [0, 1].
</details>

### Exercise 06 · Try it — Clamp01

Write a custom transform called `Clamp01` that clamps all values to [0, 1] using `torch.clamp`. Insert it into a `Compose` pipeline after `AddGaussianNoise`. Verify the output never exceeds 1.0.

In [ ]:
# your code here


<details><summary>▸ Worked answer</summary>

`class Clamp01:` with `def __call__(self, t): return torch.clamp(t, 0, 1)`. Placed after the noise step (and before `Normalize`), `out.max() <= 1` and `out.min() >= 0` always hold.
</details>

### Interview-style

(a) “What does a DataLoader do, and why would you set `shuffle=True` for training but `shuffle=False` for evaluation?” — three sentences.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

A DataLoader draws samples from a Dataset, stacks them into batches of a chosen size, and yields them one at a time, optionally loading in parallel. Shuffling during training means each epoch presents the samples in a new order and new groupings, so the model can’t latch onto the order. Evaluation only measures, so order doesn’t matter — and a fixed order makes results reproducible and debugging easier.
</details>

### Interview-style

(b) “Why do we use different transforms for training and evaluation, and what would go wrong if we used augmentation during evaluation?” — three sentences, then name three augmentations for image classification.

_Write your answer here before opening the model answer._

<details><summary>▸ Model answer</summary>

Training transforms include random augmentation so the model sees a different view of each image every epoch, which reduces overfitting. Evaluation must be deterministic and reproducible, so it uses only fixed steps such as resize, centre crop, ToTensor and Normalize. Augmenting at evaluation time makes predictions vary from run to run and adds noise to the metrics — silently. Three augmentations: `RandomHorizontalFlip`, `RandomResizedCrop`, `ColorJitter`.
</details>